# Intel Extension for PyTorch - Hello World

This notebook demonstrates the most basic usage of Intel Extension for PyTorch (IPEX).

## What is IPEX?

Intel Extension for PyTorch extends PyTorch with optimizations for Intel hardware, including:
- Intel CPUs (with AVX-512, VNNI, AMX)
- Intel GPUs (Iris Xe, Arc, Data Center GPU Max)

## Prerequisites

Make sure you have installed:
```bash
pip install torch torchvision intel-extension-for-pytorch
```

## Step 1: Import Libraries

In [1]:
import torch
import intel_extension_for_pytorch as ipex

print(f"PyTorch version: {torch.__version__}")
print(f"IPEX version: {ipex.__version__}")

[W1230 10:53:21.638810245 OperatorEntry.cpp:218] Warning: Warning only once for all operators,  other operators may also be overridden.
  Overriding a previously registered kernel for the same operator and the same dispatch key
  operator: aten::_addmm_activation(Tensor self, Tensor mat1, Tensor mat2, *, Scalar beta=1, Scalar alpha=1, bool use_gelu=False) -> Tensor
    registered at /pytorch/build/aten/src/ATen/RegisterSchema.cpp:6
  dispatch key: AutocastCPU
  previous kernel: registered at /pytorch/aten/src/ATen/autocast_mode.cpp:327
       new kernel: registered at /opt/workspace/ipex-cpu-dev/csrc/cpu/autocast/autocast_mode.cpp:112 (function operator())


PyTorch version: 2.8.0+cu128
IPEX version: 2.8.0+cpu


## Step 2: Check Available Devices

IPEX supports:
- `cpu` - Intel CPU with optimizations
- `xpu` - Intel GPU (Iris Xe, Arc, Data Center GPU)

In [2]:
# Check CPU availability (always available)
print(f"CPU available: True")

# Check Intel GPU (XPU) availability
xpu_available = hasattr(torch, 'xpu') and torch.xpu.is_available()
print(f"Intel XPU (GPU) available: {xpu_available}")

if xpu_available:
    print(f"Number of XPU devices: {torch.xpu.device_count()}")
    for i in range(torch.xpu.device_count()):
        print(f"  Device {i}: {torch.xpu.get_device_name(i)}")

# Set device (prefer XPU if available, otherwise CPU)
device = 'xpu' if xpu_available else 'cpu'
print(f"\nUsing device: {device}")

CPU available: True
Intel XPU (GPU) available: False

Using device: cpu


## Step 3: Create a Simple Tensor Operation

Let's create tensors and perform basic operations with IPEX optimization.

In [3]:
# Create tensors on the selected device
x = torch.randn(3, 3).to(device)
y = torch.randn(3, 3).to(device)

print("Tensor x:")
print(x)
print(f"\nDevice: {x.device}")

# Perform matrix multiplication
z = torch.matmul(x, y)

print("\nResult of x @ y:")
print(z)

Tensor x:
tensor([[ 1.0723, -0.7824, -1.1907],
        [ 0.2374, -1.4729, -0.7545],
        [-1.1274, -0.1870,  1.0820]])

Device: cpu

Result of x @ y:
tensor([[ 2.7168, -0.8853,  2.8318],
        [ 0.5891, -1.4030, -0.0766],
        [-3.2673,  0.3431, -3.8930]])


## Step 4: Simple Neural Network with IPEX

Let's create a minimal neural network and optimize it with IPEX.

In [ ]:
import torch.nn as nn

# Define a simple model
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.fc1 = nn.Linear(10, 50)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(50, 2)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# Create model and move to device
model = SimpleModel().to(device)
print("Model created:")
print(model)

## Step 5: Optimize Model with IPEX

IPEX provides the `optimize()` function to apply Intel-specific optimizations.

In [ ]:
# Set model to evaluation mode
model.eval()

# Create sample input
sample_input = torch.randn(1, 10).to(device)

# Optimize model with IPEX
optimized_model = ipex.optimize(model)

print("Model optimized with IPEX!")
print(f"Input shape: {sample_input.shape}")

## Step 6: Run Inference

Now let's run inference with the optimized model.

In [ ]:
# Run inference
with torch.no_grad():
    output = optimized_model(sample_input)

print("Inference output:")
print(output)
print(f"\nOutput shape: {output.shape}")
print(f"Output device: {output.device}")

## Step 7: Performance Comparison (Optional)

Let's compare the performance of the original vs IPEX-optimized model.

In [ ]:
import time

# Create a larger batch for benchmarking
batch_input = torch.randn(100, 10).to(device)

# Warmup
for _ in range(10):
    with torch.no_grad():
        _ = optimized_model(batch_input)

# Benchmark optimized model
num_iterations = 100
start_time = time.time()
for _ in range(num_iterations):
    with torch.no_grad():
        _ = optimized_model(batch_input)
end_time = time.time()

avg_time = (end_time - start_time) / num_iterations * 1000  # Convert to ms
print(f"Average inference time (IPEX optimized): {avg_time:.3f} ms")
print(f"Throughput: {num_iterations / (end_time - start_time):.2f} iterations/sec")

## Step 8: Using BFloat16 (Brain Floating Point)

IPEX supports BFloat16 for faster inference on supported Intel CPUs.

In [ ]:
# Create a new model for BF16 demonstration
model_bf16 = SimpleModel().to(device)
model_bf16.eval()

# Optimize with BFloat16 (only works on CPU with BF16 support)
if device == 'cpu':
    try:
        # Convert model to BFloat16
        optimized_model_bf16 = ipex.optimize(model_bf16, dtype=torch.bfloat16)
        
        # Create BFloat16 input
        sample_input_bf16 = torch.randn(1, 10).to(torch.bfloat16)
        
        # Run inference
        with torch.no_grad(), torch.cpu.amp.autocast():
            output_bf16 = optimized_model_bf16(sample_input_bf16)
        
        print("BFloat16 inference successful!")
        print(f"Output dtype: {output_bf16.dtype}")
        print(f"Output: {output_bf16}")
    except Exception as e:
        print(f"BFloat16 not supported on this CPU: {e}")
else:
    print("BFloat16 example is for CPU only. XPU supports FP16 instead.")

## Summary

In this notebook, you learned:

1. ✅ How to import and check IPEX version
2. ✅ How to detect available Intel devices (CPU/XPU)
3. ✅ How to create tensors on Intel devices
4. ✅ How to create a simple neural network
5. ✅ How to optimize models with `ipex.optimize()`
6. ✅ How to run inference on Intel hardware
7. ✅ How to benchmark performance
8. ✅ How to use BFloat16 for faster inference

## Next Steps

- Try with real models (ResNet, BERT, etc.)
- Explore quantization with IPEX
- Use `torch.compile()` with IPEX backend
- Check out the full IPEX documentation: https://intel.github.io/intel-extension-for-pytorch/

In [ ]:
print("🎉 Hello World with Intel Extension for PyTorch completed successfully!")